# QBronze 1.2 — Sistemas cuánticos básicos

Este cuaderno estudia, con cálculo explícito y Python, los ejercicios centrales del módulo sobre sistemas cuánticos básicos.

La idea operativa es siempre la misma:

1. Un estado cuántico se representa como un vector de amplitudes.
2. Una amplitud no es una probabilidad. La probabilidad se obtiene elevando al cuadrado el valor absoluto de la amplitud.
3. Para amplitudes reales, si

$$
|\psi\rangle = \alpha |0\rangle + \beta |1\rangle,
$$

entonces

$$
P(0)=\alpha^2,\qquad P(1)=\beta^2,\qquad \alpha^2+\beta^2=1.
$$


El cuaderno usa simulaciones sencillas con `numpy`, sin depender de Qiskit, para que todos los pasos matemáticos sean visibles.




In [1]:
import numpy as np
from math import sqrt, pi, cos, sin, log2, floor
from collections import Counter

np.set_printoptions(precision=6, suppress=True)

ket0 = np.array([1.0, 0.0])
ket1 = np.array([0.0, 1.0])

I = np.eye(2)
X = np.array([[0.0, 1.0], [1.0, 0.0]])
Z = np.array([[1.0, 0.0], [0.0, -1.0]])
H = (1 / sqrt(2)) * np.array([[1.0, 1.0], [1.0, -1.0]])

def tensor(*vectors_or_matrices):
    """Producto tensorial en el orden escrito: A ⊗ B ⊗ C."""
    result = np.array([1.0])
    for item in vectors_or_matrices:
        result = np.kron(result, item)
    return result

def norm2(state):
    """Norma al cuadrado: suma |amplitud|^2."""
    return float(np.vdot(state, state).real)

def probs(state):
    """Probabilidades de medición en la base computacional."""
    return np.abs(state) ** 2

def basis_state(bitstring):
    """Vector base |bitstring⟩ en orden lógico de izquierda a derecha."""
    n = len(bitstring)
    index = int(bitstring, 2)
    v = np.zeros(2**n)
    v[index] = 1.0
    return v

def bitstrings(n):
    return [format(i, f"0{n}b") for i in range(2**n)]

def show_state(state, n=None, tol=1e-10):
    """Representación textual compacta de un vector de estado."""
    if n is None:
        n = int(round(log2(len(state))))
    terms = []
    for amp, bits in zip(state, bitstrings(n)):
        if abs(amp) > tol:
            terms.append(f"{amp:+.6g}|{bits}⟩")
    return " ".join(terms) if terms else "0"

def sample_counts(state, shots=1024, seed=7, qiskit_order=False):
    """Muestreo de una distribución de medición. Si qiskit_order=True, invierte las etiquetas."""
    rng = np.random.default_rng(seed)
    p = probs(state)
    n = int(round(log2(len(state))))
    outcomes = rng.choice(2**n, p=p/p.sum(), size=shots)
    labels = []
    for i in outcomes:
        bits = format(i, f"0{n}b")
        labels.append(bits[::-1] if qiskit_order else bits)
    return dict(sorted(Counter(labels).items()))

def is_valid_quantum_state(v, tol=1e-9):
    return abs(norm2(np.asarray(v, dtype=float)) - 1.0) < tol

## 1. Vectores que sí representan estados cuánticos

Un vector representa un estado cuántico válido si su norma al cuadrado es exactamente 1:

$$
\sum_i |\alpha_i|^2=1.
$$

No se exige que las amplitudes sean positivas. Las amplitudes pueden ser negativas. Lo que debe ser no negativo son las probabilidades, porque se calculan como cuadrados.

Ejemplo:

$$
v=\left(\frac12,\frac12,\frac12,\frac12\right).
$$

Entonces

$$
\|v\|^2 = \left(\frac12\right)^2+\left(\frac12\right)^2+\left(\frac12\right)^2+\left(\frac12\right)^2
=\frac14+\frac14+\frac14+\frac14=1.
$$

Por tanto, es un estado cuántico válido.

In [1]:
vectors = {
    "(1/2, 1/2, 1/2, 1/2)": np.array([1/2, 1/2, 1/2, 1/2]),
    "(2/3, 1/3, 2/3, 0)": np.array([2/3, 1/3, 2/3, 0]),
    "(1/2, 0, 1/2, 0)": np.array([1/2, 0, 1/2, 0]),
    "(1/sqrt(2), -1/sqrt(2), 0, 0)": np.array([1/sqrt(2), -1/sqrt(2), 0, 0]),
    "(1/sqrt(2), -1/sqrt(2), -1/sqrt(2), -1/sqrt(2))": np.array([1/sqrt(2), -1/sqrt(2), -1/sqrt(2), -1/sqrt(2)]),
    "(1/2, -1/2, -1/2, 1/2)": np.array([1/2, -1/2, -1/2, 1/2]),
}

for name, v in vectors.items():
    print(f"{name:58s} norma^2 = {norm2(v):.6f}  válido = {is_valid_quantum_state(v)}")

NameError: name 'np' is not defined

## 2. Estados que no son válidos

Para descartar un vector, basta con calcular

$$
\sum_i |\alpha_i|^2.
$$

Si el resultado no es 1, el vector no está normalizado. Esto no significa que el vector sea inútil: puede normalizarse dividiéndolo entre su norma. Pero como está escrito, no representa directamente un estado físico normalizado.

In [ ]:
not_candidates = {
    "(1/4, -1/4, -1/4, 1/4)": np.array([1/4, -1/4, -1/4, 1/4]),
    "(2/3, -1/3, 2/3, 0)": np.array([2/3, -1/3, 2/3, 0]),
    "(1/sqrt(2), -1/sqrt(2), 0, 0)": np.array([1/sqrt(2), -1/sqrt(2), 0, 0]),
    "(1/sqrt(2), -1/sqrt(2), -1/sqrt(2), -1/sqrt(2))": np.array([1/sqrt(2), -1/sqrt(2), -1/sqrt(2), -1/sqrt(2)]),
    "(1/2, 0, 1/2, 0)": np.array([1/2, 0, 1/2, 0]),
    "(0, -1, 0, 0)": np.array([0, -1, 0, 0]),
}

for name, v in not_candidates.items():
    print(f"{name:62s} norma^2 = {norm2(v):.6f}  no válido = {not is_valid_quantum_state(v)}")

## 3. La compuerta Hadamard sobre $|1\rangle$

La matriz de Hadamard es

$$
H=\frac{1}{\sqrt{2}}
\begin{pmatrix}
1 & 1\\
1 & -1
\end{pmatrix}.
$$

El estado $|1\rangle$ es

$$
|1\rangle=\begin{pmatrix}0\\1\end{pmatrix}.
$$

Multiplicamos sin saltar pasos:

$$
H|1\rangle
=\frac{1}{\sqrt{2}}
\begin{pmatrix}
1 & 1\\
1 & -1
\end{pmatrix}
\begin{pmatrix}0\\1\end{pmatrix}
=\frac{1}{\sqrt{2}}
\begin{pmatrix}
1\cdot 0 + 1\cdot 1\\
1\cdot 0 + (-1)\cdot 1
\end{pmatrix}
=\frac{1}{\sqrt{2}}
\begin{pmatrix}1\\-1\end{pmatrix}.
$$

Por definición,

$$
|-\rangle=\frac{|0\rangle-|1\rangle}{\sqrt{2}}.
$$

Entonces:

$$
H|1\rangle=|-\rangle.
$$

In [ ]:
result = H @ ket1
print("H|1> =", result)
print("Forma ket:", show_state(result, n=1))
print("Probabilidades:", probs(result))

## 4. Qubit real parametrizado por un ángulo

Para un qubit real se usa frecuentemente la parametrización

$$
|\psi(x)\rangle=\cos(x)|0\rangle+\sin(x)|1\rangle.
$$

La amplitud de $|0\rangle$ es $\cos(x)$. La probabilidad de observar $|0\rangle$ no es $\cos(x)$, sino

$$
P(0)=|\cos(x)|^2=\cos^2(x).
$$

Análogamente,

$$
P(1)=\sin^2(x).
$$

In [ ]:
for x in [0, pi/6, pi/4, pi/3, pi/2]:
    state = np.array([cos(x), sin(x)])
    print(f"x = {x:.4f} rad")
    print("estado =", state)
    print("P(0) = cos^2(x) =", cos(x)**2)
    print("P(1) = sin^2(x) =", sin(x)**2)
    print("suma =", cos(x)**2 + sin(x)**2)
    print("---")

## 5. Amplitud frente a probabilidad

Si un qubit está en el estado

$$
|\psi\rangle=0.43|0\rangle-0.90|1\rangle,
$$

entonces la amplitud asociada a $|1\rangle$ es $-0.90$. La probabilidad de observar $|1\rangle$ es

$$
P(1)=|-0.90|^2=0.81.
$$

La pregunta por la amplitud y la pregunta por la probabilidad no son equivalentes.

In [ ]:
psi = np.array([0.43, -0.90])
print("amplitud de |0>:", psi[0])
print("amplitud de |1>:", psi[1])
print("probabilidad de |0>:", psi[0]**2)
print("probabilidad de |1>:", psi[1]**2)
print("norma^2 aproximada:", norm2(psi))

## 6. Compuerta NOT cuántica: la compuerta $X$

La compuerta NOT cuántica es la matriz

$$
X=\begin{pmatrix}
0&1\\
1&0
\end{pmatrix}.
$$

Actúa así:

$$
X|0\rangle=|1\rangle,\qquad X|1\rangle=|0\rangle.
$$

Por tanto, si el circuito empieza en $|0\rangle$, se aplica $X$, y luego se mide, el resultado debe ser $1$ con probabilidad 1.

In [ ]:
state = ket0.copy()
state = X @ state
print("Estado después de X|0>:", show_state(state, n=1))
print("Probabilidades:", {"0": probs(state)[0], "1": probs(state)[1]})
print("Muestreo:", sample_counts(state, shots=1024, seed=1))

## 7. Preparar $|-\rangle$ con $X$ seguido de $H$

Partimos de $|0\rangle$. Primero aplicamos $X$:

$$
X|0\rangle=|1\rangle.
$$

Luego aplicamos $H$:

$$
H|1\rangle=\frac{|0\rangle-|1\rangle}{\sqrt{2}}=|-\rangle.
$$

Por tanto, la secuencia es:

$$
|0\rangle \xrightarrow{X} |1\rangle \xrightarrow{H} |-\rangle.
$$

In [ ]:
state = ket0.copy()
state = X @ state
state = H @ state
print("Estado final:", show_state(state, n=1))
print("Vector:", state)
print("Probabilidades:", probs(state))

## 8. Orden de bits al imprimir resultados de dos qubits

En muchos entornos de circuitos cuánticos, cuando se mide $q[0]$ en $c[0]$ y $q[1]$ en $c[1]$, la cadena impresa aparece como

$$
c[1]c[0].
$$

Por eso, si se aplica $X$ al qubit $q[0]$ y se mide, el estado lógico interno es $q[1]q[0]=01$, y el conteo impreso aparece como `'01'`.

La clave práctica es no confundir el índice del qubit con la posición visual de la cadena de salida.

In [ ]:
# Estado lógico en orden |q1 q0>. Aplicar X a q0 produce |01>.
state_q1q0 = basis_state("01")
print("Estado lógico |q1 q0>:", show_state(state_q1q0, n=2))
print("Conteo con etiquetas en orden lógico:", sample_counts(state_q1q0, shots=100, seed=2, qiskit_order=False))

# Si ya usamos |q1 q0>, la cadena coincide con c1c0.
# El resultado impreso esperado es {'01': 100}.

## 9. Divisor de haz ideal como moneda cuántica

En el modelo ideal de moneda cuántica, un fotón que atraviesa un divisor de haz balanceado tiene dos posibles salidas:

$$
P(\text{reflejado})=\frac12,\qquad P(\text{transmitido})=\frac12.
$$

Una forma abstracta de modelarlo es análoga a aplicar $H$ sobre un estado base: el resultado queda con dos amplitudes de magnitud $1/\sqrt{2}$, y por la regla de Born las probabilidades son $1/2$ y $1/2$.

In [ ]:
state = H @ ket0
print("Estado abstracto después de H|0>:", show_state(state, n=1))
print("Probabilidad salida 0:", probs(state)[0])
print("Probabilidad salida 1:", probs(state)[1])
print("Muestreo aproximado:", sample_counts(state, shots=1000, seed=3))

## 10. Resumen operativo del módulo

Reglas que conviene recordar:

$$
\text{estado válido} \iff \sum_i |\alpha_i|^2=1.
$$

$$
P(i)=|\alpha_i|^2.
$$

$$
H|0\rangle=|+\rangle=\frac{|0\rangle+|1\rangle}{\sqrt{2}}.
$$

$$
H|1\rangle=|-\rangle=\frac{|0\rangle-|1\rangle}{\sqrt{2}}.
$$

$$
X|0\rangle=|1\rangle,\qquad X|1\rangle=|0\rangle.
$$